In [150]:
# EDA
import pandas as pd
import plotly.express as px

# ML
from sklearn.datasets import load_iris
from sklearn.mixture import GaussianMixture

# HP
import optuna

### Carga de Dados

In [151]:
iris = load_iris()

In [152]:
# Transforma iris em um DataFrame
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)

In [153]:
df_iris['target'] = iris.target

In [154]:
df_iris.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


In [155]:
df_iris.head(20)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
5,5.4,3.9,1.7,0.4,0
6,4.6,3.4,1.4,0.3,0
7,5.0,3.4,1.5,0.2,0
8,4.4,2.9,1.4,0.2,0
9,4.9,3.1,1.5,0.1,0


In [156]:
df_iris.tail(20)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
130,7.4,2.8,6.1,1.9,2
131,7.9,3.8,6.4,2.0,2
132,6.4,2.8,5.6,2.2,2
133,6.3,2.8,5.1,1.5,2
134,6.1,2.6,5.6,1.4,2
135,7.7,3.0,6.1,2.3,2
136,6.3,3.4,5.6,2.4,2
137,6.4,3.1,5.5,1.8,2
138,6.0,3.0,4.8,1.8,2
139,6.9,3.1,5.4,2.1,2


### EDA

In [157]:
df_iris.describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


In [158]:
df_iris.shape

(150, 5)

In [159]:
df_iris.target.value_counts()

target
0    50
1    50
2    50
Name: count, dtype: int64

In [160]:
df_iris.target.value_counts(normalize=True)

target
0    0.333333
1    0.333333
2    0.333333
Name: proportion, dtype: float64

In [161]:
X = df_iris.drop('target', axis=1)
y = df_iris['target']

In [162]:
def gmm_objective(trial):
    n_components = trial.suggest_int('n_components', 3, 10)
    covariance_type = trial.suggest_categorical('covariance_type', ['full', 'tied', 'diag', 'spherical'])

    gmm = GaussianMixture(n_components=n_components, covariance_type=covariance_type, random_state=42)

    gmm.fit(X)

    bic_gmm = gmm.bic(X)

    return bic_gmm

In [163]:
search_space = {'n_components': [3, 4, 5, 6, 7, 8, 9, 10],
                'covariance_type': ['full', 'tied', 'diag', 'spherical']}

sampler = optuna.samplers.GridSampler(search_space=search_space)

estudo_gmm = optuna.create_study(direction='maximize', sampler=sampler)

[I 2026-06-06 14:36:39,663] A new study created in memory with name: no-name-935d223c-76b5-4819-8507-f57b7394081b


In [164]:
estudo_gmm.optimize(gmm_objective, n_trials=32)

[I 2026-06-06 14:36:39,681] Trial 0 finished with value: 625.3638032146685 and parameters: {'n_components': 6, 'covariance_type': 'tied'}. Best is trial 0 with value: 625.3638032146685.
[I 2026-06-06 14:36:39,689] Trial 1 finished with value: 713.7282960545704 and parameters: {'n_components': 9, 'covariance_type': 'diag'}. Best is trial 1 with value: 713.7282960545704.
[I 2026-06-06 14:36:39,696] Trial 2 finished with value: 607.1564871830387 and parameters: {'n_components': 5, 'covariance_type': 'tied'}. Best is trial 1 with value: 713.7282960545704.
[I 2026-06-06 14:36:39,708] Trial 3 finished with value: 681.458986569831 and parameters: {'n_components': 5, 'covariance_type': 'full'}. Best is trial 1 with value: 713.7282960545704.
[I 2026-06-06 14:36:39,712] Trial 4 finished with value: 744.6403610652238 and parameters: {'n_components': 3, 'covariance_type': 'diag'}. Best is trial 4 with value: 744.6403610652238.
[I 2026-06-06 14:36:39,724] Trial 5 finished with value: 640.4607608014

In [165]:
best_params = estudo_gmm.best_params

In [166]:
best_gmm = GaussianMixture(n_components=best_params['n_components'], covariance_type=best_params['covariance_type'], random_state=42)

best_gmm.fit(X)

best_bic = best_gmm.bic(X)

In [167]:
print("Quantidade ideal de componentes: ", best_params['n_components'])
print("Tipo de Covariância: ", best_params['covariance_type'])
print("BIC do melhor modelo: ", best_bic)

Quantidade ideal de componentes:  10
Tipo de Covariância:  full
BIC do melhor modelo:  905.2282885045927


In [168]:
clusters_gmm = best_gmm.predict(X)

In [169]:
clusters_gmm

array([6, 6, 6, 6, 6, 1, 6, 6, 6, 6, 6, 6, 6, 6, 1, 1, 1, 1, 1, 1, 6, 1,
       6, 1, 6, 6, 1, 6, 6, 6, 6, 1, 6, 1, 6, 6, 1, 6, 6, 6, 6, 6, 6, 1,
       1, 6, 6, 6, 6, 6, 0, 5, 0, 5, 0, 5, 5, 4, 0, 4, 4, 5, 5, 5, 4, 0,
       5, 5, 5, 4, 7, 0, 7, 5, 0, 0, 0, 0, 5, 4, 4, 4, 4, 7, 5, 5, 0, 5,
       5, 4, 5, 5, 5, 4, 5, 5, 5, 5, 4, 5, 3, 7, 3, 9, 3, 8, 4, 8, 9, 2,
       9, 9, 3, 7, 7, 3, 9, 8, 8, 7, 3, 7, 8, 7, 3, 8, 7, 7, 9, 8, 8, 8,
       9, 7, 9, 8, 3, 9, 7, 3, 3, 9, 7, 3, 3, 9, 7, 9, 3, 7])

In [170]:
clusters_gmm_prob = best_gmm.predict_proba(X)

In [171]:
clusters_gmm_prob


array([[8.23319309e-063, 4.05745296e-002, 0.00000000e+000, ...,
        1.39494726e-271, 8.74692949e-052, 1.81158191e-171],
       [1.84122718e-044, 2.18833912e-003, 0.00000000e+000, ...,
        3.48126043e-288, 5.13038023e-053, 1.62582414e-192],
       [8.96507941e-062, 2.55383663e-004, 0.00000000e+000, ...,
        1.92671201e-307, 8.69294362e-060, 7.14310533e-198],
       ...,
       [3.94162081e-007, 4.69846517e-171, 0.00000000e+000, ...,
        5.04796002e-005, 1.58914458e-015, 9.87392693e-001],
       [9.30347217e-035, 1.04244512e-178, 0.00000000e+000, ...,
        2.71748309e-009, 3.14816097e-037, 2.25661574e-004],
       [2.79310537e-032, 8.36237198e-134, 0.00000000e+000, ...,
        9.92334205e-001, 2.45682227e-029, 6.55913792e-004]],
      shape=(150, 10))

In [172]:
df_iris['cluster'] = clusters_gmm.astype(int)

In [173]:
df_iris.head(10)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,cluster
0,5.1,3.5,1.4,0.2,0,6
1,4.9,3.0,1.4,0.2,0,6
2,4.7,3.2,1.3,0.2,0,6
3,4.6,3.1,1.5,0.2,0,6
4,5.0,3.6,1.4,0.2,0,6
5,5.4,3.9,1.7,0.4,0,1
6,4.6,3.4,1.4,0.3,0,6
7,5.0,3.4,1.5,0.2,0,6
8,4.4,2.9,1.4,0.2,0,6
9,4.9,3.1,1.5,0.1,0,6


In [182]:
px.scatter(df_iris, x='sepal width (cm)', y='sepal length (cm)', color='cluster')

In [181]:
px.scatter(df_iris, x='petal width (cm)', y='petal length (cm)', color='cluster')

In [180]:
px.scatter(df_iris, x='petal width (cm)', y='sepal length (cm)', color='cluster')

In [179]:
px.scatter(df_iris, x='sepal width (cm)', y='petal length (cm)', color='cluster')